# Kohonen Network (SOM) — Europe Dataset
**Sistemas de Inteligencia Artificial 2026 — TP4, Exercice 1**

## Objective
Apply a Self-Organizing Map (Kohonen network) to the `europe.csv` dataset to:
1. Associate countries with similar geopolitical, economic and social characteristics
2. Visualize the resulting map
3. Plot the average distances between neighboring neurons (U-Matrix)
4. Analyze how many countries are assigned to each neuron

## Theory (§14 of course)
A Kohonen network is a **competitive unsupervised** learning algorithm. It maps high-dimensional data onto a 2D grid where **similar inputs activate neighboring neurons** — this is the "winner take all" mechanism.

**Key algorithm steps:**
- Select input `X^p`
- Find winning neuron `k̂ = argmin ||X^p - W_j||` (Euclidean distance)
- Update `k̂` and its neighborhood: `W_j^{t+1} = W_j^t + η(t) * (X^p - W_j^t)`
- Decay η(t) and R(t) over time


## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

## 2. Load & Standardize Data

In [ ]:
df = pd.read_csv('europe.csv')
countries     = df['Country'].values
features      = df.drop(columns='Country')
feature_names = features.columns.tolist()

# Standardization is mandatory (§14: "The network requires standardizing the variables")
# Without it, Area (km²) would dominate GDP ($/cap) purely because of its scale.
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(features)

print(f"Dataset: {X_scaled.shape[0]} countries, {X_scaled.shape[1]} features")
print(f"Features: {feature_names}")
print(f"Mean after scaling ≈ {X_scaled.mean(axis=0).round(10)}")
print(f"Std  after scaling = {X_scaled.std(axis=0).round(10)}")

## 3. Grid Size Choice — Motivated by PCA Results

### Why 4×2 and not 5×5?

Before designing the SOM, we already performed a PCA on this dataset. The results were:

| Component | Explained variance |
|---|---|
| PC1 | **46.1%** |
| PC1 + PC2 | **63.1%** |
| PC1 + PC2 + PC3 | 78.3% |

**Key insight:** nearly half the variance is captured by a single axis (PC1), which already orders the 28 countries along a clear socio-economic prosperity dimension — from Luxembourg and Switzerland down to Ukraine and Bulgaria. The data is **quasi-unidimensional** in its dominant structure, with a weaker secondary axis (PC2, 17%).

This has direct consequences for the SOM grid design:

**Against 5×5 (25 neurons):**
- 25 neurons for 28 countries → ~1 country per neuron → most cells would contain 0 or 1 country
- No meaningful groupings would emerge — the map would just replicate the input list
- The SOM's value is in clustering; with 25 cells it adds no information over the raw data

**In favor of 4×2 (8 neurons):**
- 8 neurons for 28 countries → ~3.5 countries per neuron → meaningful, interpretable groups
- The **4-column axis** naturally maps onto the PC1 prosperity gradient (4 levels of development)
- The **2-row axis** captures the secondary PC2 dimension (social tensions vs. stability)
- This aligns the SOM topology with the actual variance structure of the data

**Formal justification:**
The empirical rule `√(5×N) ≈ √140 ≈ 12` suggests ~12 neurons when data is truly multidimensional.
Since PCA reveals that most variance is already explained by 1-2 axes, we can go well below this — 8 neurons is appropriate.

**Alternative considered:** 3×3 (9 neurons). Rejected because the rectangular 4×2 grid better reflects
the asymmetry between PC1 (dominant) and PC2 (secondary), and its orientation is more interpretable.


## 4. Kohonen SOM Implementation

### Parameter choices and justifications

| Parameter | Value | Justification |
|---|---|---|
| Grid size | **4×2** | See §3 above — motivated by PCA structure |
| Topology | Rectangular | Sufficient for tabular data; matches the PC1/PC2 axes |
| Initialization | Random samples from training set | Course §14: avoids "dead units" — neurons initialized with random values may never win |
| Initial R(0) | **grid_size = 4** | Course §14: "R(0) can be the total size of the network" — full grid size so every neuron is in the neighborhood at t=0 |
| Initial η(0) | 0.5 | Course §14: η(0) < 1; 0.5 allows fast early learning |
| η decay | **Exponential:** η(t) = η₀·exp(-t/T) | Fast initial learning that slows down smoothly — standard in SOM literature |
| R decay | **R(t) = 1+(R₀-1)·exp(-t/T)** | Converges to R=1 as t→∞ (course §14: "R(t)→1") — only immediate neighbors updated at end of training |
| Iterations | 500 × n_features = 3500 | Course §14: "500×n" — sufficient for convergence on 28 samples |


In [ ]:
class KohonenSOM:
    def __init__(self, grid_rows=2, grid_cols=4, n_features=7,
                 n_iterations=3500, eta0=0.5, R0=None, random_state=42):
        self.grid_rows    = grid_rows
        self.grid_cols    = grid_cols
        self.n_neurons    = grid_rows * grid_cols
        self.n_features   = n_features
        self.n_iterations = n_iterations
        self.eta0         = eta0
        # R(0) = full grid size (largest dimension) — course §14
        self.R0           = R0 if R0 else float(max(grid_rows, grid_cols))
        self.T            = n_iterations / 2   # decay time constant
        self.rng          = np.random.RandomState(random_state)
        self.weights      = None  # shape: (grid_rows, grid_cols, n_features)

    def _grid_positions(self):
        """All neuron (row, col) positions as an array."""
        return np.array([[i, j]
                         for i in range(self.grid_rows)
                         for j in range(self.grid_cols)])

    def _find_bmu(self, x):
        """Best Matching Unit: neuron whose weight is closest to x."""
        diff  = self.weights.reshape(-1, self.n_features) - x
        dists = np.linalg.norm(diff, axis=1)
        flat  = np.argmin(dists)
        return divmod(flat, self.grid_cols)

    # Decay functions
    def _eta(self, t):
        """Exponential decay: η(t) = η₀·exp(-t/T)"""
        return self.eta0 * np.exp(-t / self.T)

    def _R(self, t):
        """Decay toward 1: R(t) = 1 + (R₀-1)·exp(-t/T)
        At t=0: R=R₀ (full neighborhood). At t→∞: R→1 (immediate neighbors only).
        Course §14: R(t)→1 as t→∞"""
        return 1 + (self.R0 - 1) * np.exp(-t / self.T)

    def _h(self, neuron_pos, bmu_pos, t):
        """Gaussian neighborhood function."""
        dist_sq = np.sum((neuron_pos - bmu_pos) ** 2)
        R = self._R(t)
        return np.exp(-dist_sq / (2 * R ** 2))

    def fit(self, X):
        n_samples = X.shape[0]
        # Initialize weights with samples from training data (avoids dead units)
        idx = self.rng.choice(n_samples, size=self.n_neurons, replace=True)
        self.weights = X[idx].reshape(
            self.grid_rows, self.grid_cols, self.n_features).copy()

        grid_pos = self._grid_positions()

        for t in range(self.n_iterations):
            x = X[self.rng.randint(0, n_samples)]
            bmu_row, bmu_col = self._find_bmu(x)
            bmu_pos = np.array([bmu_row, bmu_col])
            eta_t = self._eta(t)
            for flat_idx, pos in enumerate(grid_pos):
                h = self._h(pos, bmu_pos, t)
                r, c = divmod(flat_idx, self.grid_cols)
                self.weights[r, c] += eta_t * h * (x - self.weights[r, c])
        return self

    def predict(self, X):
        return [self._find_bmu(x) for x in X]

    def u_matrix(self):
        """Average distance between each neuron and its direct neighbors."""
        U = np.zeros((self.grid_rows, self.grid_cols))
        for i in range(self.grid_rows):
            for j in range(self.grid_cols):
                neighbors = []
                for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
                    ni, nj = i+di, j+dj
                    if 0 <= ni < self.grid_rows and 0 <= nj < self.grid_cols:
                        d = np.linalg.norm(self.weights[i,j] - self.weights[ni,nj])
                        neighbors.append(d)
                U[i, j] = np.mean(neighbors)
        return U

## 5. Train the SOM

In [ ]:
som = KohonenSOM(grid_rows=2, grid_cols=4,
                n_features=X_scaled.shape[1],
                n_iterations=3500, eta0=0.5)
som.fit(X_scaled)
assignments = som.predict(X_scaled)

print("Training complete.")
print(f"Grid: {som.grid_rows}×{som.grid_cols} = {som.n_neurons} neurons")
print(f"R(0) = {som.R0}  (full grid size — all neurons in neighborhood at start)")
print(f"η(0) = {som.eta0}")
print()
print(f"Decay at t=T/2 (midpoint):")
print(f"  η(T/2) = {som._eta(som.T//2):.4f}  (exponential)")
print(f"  R(T/2) = {som._R(som.T//2):.4f}   (polynomial — stays larger longer)")
print()
print(f"Decay at t=T (end):")
print(f"  η(T)   = {som._eta(som.T):.4f}")
print(f"  R(T)   = {som._R(som.T):.4f}")

## 6. Visualization 1 — Country Map

Each cell shows countries assigned to that neuron.
The 4 columns should reflect the PC1 prosperity gradient (left = low, right = high).
The 2 rows reflect the PC2 secondary dimension.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(-0.5, som.grid_cols - 0.5)
ax.set_ylim(-0.5, som.grid_rows - 0.5)
ax.set_facecolor('#EEF3F7')
ax.set_title('Kohonen SOM (4×2) — Country Assignments\n'
             'Columns reflect PC1 (prosperity gradient), Rows reflect PC2',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Column  (← lower prosperity  |  higher prosperity →)')
ax.set_ylabel('Row')
ax.set_xticks(range(som.grid_cols))
ax.set_yticks(range(som.grid_rows))

assignment_map = {}
for country, (r, c) in zip(countries, assignments):
    assignment_map.setdefault((r, c), []).append(country)

for i in range(som.grid_rows):
    for j in range(som.grid_cols):
        n_c = len(assignment_map.get((i, j), []))
        intensity = min(n_c / 5, 1.0)
        color = plt.cm.Blues(0.15 + 0.7 * intensity)
        rect = plt.Rectangle([j - 0.48, i - 0.48], 0.96, 0.96,
                              facecolor=color, edgecolor='white', linewidth=2.5)
        ax.add_patch(rect)
        label = '\n'.join(assignment_map.get((i, j), ['—']))
        ax.text(j, i, label, ha='center', va='center', fontsize=8,
                color='#1A2233' if n_c < 3 else 'white', fontweight='bold')

# Add PC1 arrow annotation
ax.annotate('', xy=(som.grid_cols-0.5, -0.7), xytext=(-0.5, -0.7),
            xycoords='data', textcoords='data',
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=2),
            annotation_clip=False)
ax.text(som.grid_cols/2 - 0.5, -0.85, 'PC1: Prosperity →',
        ha='center', fontsize=9, color='steelblue', style='italic',
        annotation_clip=False)

plt.tight_layout()
plt.savefig('som_map.png', bbox_inches='tight')
plt.show()

print("Countries per cell:")
for (r,c), ctries in sorted(assignment_map.items()):
    print(f"  [{r},{c}]: {', '.join(ctries)}")

### Comments on the map
- The **4-column structure** should reveal a gradient from economically weaker countries (left) to stronger ones (right), mirroring PC1.
- Countries in the **same cell** share the closest profile across all 7 variables — the SOM found this grouping without any label.
- With ~3-4 countries per cell on average, the groupings are **meaningful and interpretable**, unlike a 5×5 grid which would produce mostly empty cells.
- Compare these groupings to the PC1 ranking from the PCA notebook — they should largely agree, validating both analyses.


## 7. Visualization 2 — U-Matrix (Neuron Distance Map)

The U-Matrix shows the **average Euclidean distance** between each neuron and its direct neighbors.
- **Dark (low value)** → neurons are close → cluster interior
- **Light (high value)** → neurons are far → cluster boundary


In [ ]:
U = som.u_matrix()

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(U, cmap='RdYlGn_r', interpolation='nearest', aspect='auto')
plt.colorbar(im, ax=ax, label='Mean distance to neighbors')
ax.set_title('U-Matrix — Average Distance Between Neighboring Neurons',
             fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('Column')
ax.set_ylabel('Row')

for i in range(som.grid_rows):
    for j in range(som.grid_cols):
        ax.text(j, i, f'{U[i,j]:.2f}', ha='center', va='center',
                fontsize=11, color='black', fontweight='bold')

plt.tight_layout()
plt.savefig('u_matrix.png', bbox_inches='tight')
plt.show()

print(f"Min distance: {U.min():.3f}  → cluster interior")
print(f"Max distance: {U.max():.3f}  → cluster boundary")
print(f"Mean: {U.mean():.3f}")

### Interpretation
- **High U-Matrix values** between columns indicate the main split between country groups — for example, the boundary between Eastern and Western European clusters.
- **Low values** within a row confirm that neighboring neurons represent similar country profiles.
- With only 8 neurons, each boundary carries more information than in a 5×5 grid where most boundaries would be noise.


## 8. Visualization 3 — Activation Count per Neuron


In [ ]:
activation_count = np.zeros((som.grid_rows, som.grid_cols), dtype=int)
for r, c in assignments:
    activation_count[r, c] += 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im = axes[0].imshow(activation_count, cmap='YlOrRd',
                    interpolation='nearest', aspect='auto')
plt.colorbar(im, ax=axes[0], label='Countries assigned')
axes[0].set_title('Activation Count per Neuron', fontweight='bold')
axes[0].set_xlabel('Column')
axes[0].set_ylabel('Row')
for i in range(som.grid_rows):
    for j in range(som.grid_cols):
        axes[0].text(j, i, str(activation_count[i,j]),
                     ha='center', va='center', fontsize=14,
                     color='black' if activation_count[i,j] < 4 else 'white',
                     fontweight='bold')

unique_counts, freq = np.unique(activation_count, return_counts=True)
axes[1].bar(unique_counts, freq, color='steelblue', edgecolor='white', width=0.6)
axes[1].set_title('Distribution of Countries per Neuron', fontweight='bold')
axes[1].set_xlabel('Countries assigned to neuron')
axes[1].set_ylabel('Number of neurons')
axes[1].set_xticks(unique_counts)
for x, y in zip(unique_counts, freq):
    axes[1].text(x, y + 0.05, str(y), ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('activation_count.png', bbox_inches='tight')
plt.show()

n_active = np.sum(activation_count > 0)
n_dead   = np.sum(activation_count == 0)
print(f"Active neurons : {n_active}/{som.n_neurons}")
print(f"Dead neurons   : {n_dead}")
print(f"Max per neuron : {activation_count.max()}")
print(f"Mean per active: {28/n_active:.1f}")

### Comments
- With 8 neurons and 28 countries, we expect ~3-4 countries per neuron — a healthy, balanced distribution.
- **Dead neurons** (0 countries) would indicate the grid is too large — this should not happen with our 4×2 choice.
- A neuron with 6+ countries would suggest the grid is too small in that region — consider a 4×3 as an alternative.
- The course notes: "the grid size must be decided from the start and there is no proven criterion" — our PCA-motivated choice is the best justification available.


## 9. Decay Curves — Visualization

It is useful to plot how η(t) and R(t) evolve during training to confirm the decay behavior.


In [ ]:
t_vals = np.arange(som.n_iterations)
eta_vals = np.array([som._eta(t) for t in t_vals])
R_vals   = np.array([som._R(t)   for t in t_vals])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(t_vals, eta_vals, color='steelblue', linewidth=2)
axes[0].set_title('Learning Rate η(t) — Exponential Decay', fontweight='bold')
axes[0].set_xlabel('Iteration t')
axes[0].set_ylabel('η(t)')
axes[0].axvline(som.T, color='gray', linestyle='--', label=f't=T={int(som.T)}')
axes[0].legend()
axes[0].set_yscale('log')

axes[1].plot(t_vals, R_vals, color='coral', linewidth=2)
axes[1].set_title('Neighborhood Radius R(t) — Polynomial Decay', fontweight='bold')
axes[1].set_xlabel('Iteration t')
axes[1].set_ylabel('R(t)')
axes[1].axhline(1.0, color='gray', linestyle='--', label='R=1 (immediate neighbors only)')
axes[1].axvline(som.T, color='gray', linestyle=':', label=f't=T={int(som.T)}')
axes[1].legend()

plt.tight_layout()
plt.savefig('decay_curves.png', bbox_inches='tight')
plt.show()

print("Why exponential for η and polynomial for R?")
print("  η: fast decay ensures learning rate drops quickly → fine-tuning in later iterations")
print("  R: slower polynomial decay keeps the neighborhood large for longer → better")
print("     global topology organization before focusing on local adjustments")

## 10. Summary

| Parameter | Value | Motivation |
|---|---|---|
| Grid | 4×2 | PCA shows quasi-1D structure; 8 neurons → ~3-4 countries/cell |
| R(0) | 4 (grid size) | Full neighborhood at start — all neurons learn together |
| η(0) | 0.5 | Fast initial learning, stable convergence |
| η decay | Exponential | Quick slowdown → fine-tuning phase |
| R decay | 1+(R₀-1)·exp(-t/T) | Converges to R=1 — immediate neighbors only at end (course §14) |
| Iterations | 3500 | 500 × n_features (course §14) |

**Connection to PCA:** The SOM groupings should reflect the PC1 axis — Western/Northern European countries (Luxembourg, Norway, Switzerland) should cluster on one end, Eastern European countries (Ukraine, Bulgaria) on the other. The SOM validates the PCA result using a completely different, biologically-inspired algorithm.
